# Public Benchmark Baseline: CoLES, CoLES + Head, CoLES + CatBoost


In [ ]:
# Cell 1 - Setup & Install
import json
import os
import pickle
import shutil
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import yaml
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyyaml", "-q"])
    import yaml

REPO_DIR = Path("/kaggle/working/denoising-event-sequences")
if not REPO_DIR.exists():
    REPO_DIR = Path.cwd()
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

DATA_ROOT = Path("/kaggle/working/data")
RAW_ROOT = DATA_ROOT / "raw"
PROCESSED_ROOT = DATA_ROOT / "processed"
OUTPUT_ROOT = Path("/kaggle/working/outputs/public_benchmarks")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def run_logged(cmd, log_path):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("Running:", " ".join(map(str, cmd)))
    with log_path.open("w") as log_file:
        proc = subprocess.Popen(
            [str(x) for x in cmd],
            cwd=str(REPO_DIR),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in proc.stdout:
            print(line, end="")
            log_file.write(line)
        return_code = proc.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)
    print(f"Log saved: {log_path}")

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, Dataset
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch", "-q"])
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, Dataset

try:
    from catboost import CatBoostClassifier, Pool
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "catboost", "-q"])
    from catboost import CatBoostClassifier, Pool

try:
    import ptls  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pytorch-lifestream", "-q"])


In [ ]:
# Shared - Prepare Public Benchmark Data If Missing
DATASET_NAME = "gender"  # "gender" or "age_group"
DATASET_CONFIG_PATHS = {
    "gender": REPO_DIR / "configs" / "datasets" / "gender.yaml",
    "age_group": REPO_DIR / "configs" / "datasets" / "age_group.yaml",
}
PROCESSED_DIR = PROCESSED_ROOT / f"{DATASET_NAME}_benchmark"
RUN_PREPARE_DATA_IF_MISSING = True
FORCE_REBUILD_PROCESSED = False
MAX_SEQ_LEN = 512

required_artifacts = ["events.parquet", "canonical_events.parquet", "transformed_events.parquet", "splits.json", "preprocessor.pkl", "prepared_config.yaml", "data_report.json"]
artifacts_exist = all((PROCESSED_DIR / name).exists() for name in required_artifacts)
if artifacts_exist:
    with (PROCESSED_DIR / "data_report.json").open() as f:
        existing_report = json.load(f)
    artifacts_exist = bool(existing_report.get("events_are_raw_pretransform", False))
if FORCE_REBUILD_PROCESSED and PROCESSED_DIR.exists():
    shutil.rmtree(PROCESSED_DIR)
    artifacts_exist = False

if RUN_PREPARE_DATA_IF_MISSING and not artifacts_exist:
    run_logged(
        [
            sys.executable,
            REPO_DIR / "scripts" / "prepare_public_benchmark_data.py",
            "--dataset", DATASET_NAME,
            "--raw-root", RAW_ROOT,
            "--output-dir", PROCESSED_DIR,
            "--max-seq-len", MAX_SEQ_LEN,
        ],
        OUTPUT_ROOT / "logs" / f"{DATASET_NAME}_prepare_public_benchmark_data.log",
    )

assert all((PROCESSED_DIR / name).exists() for name in required_artifacts), PROCESSED_DIR
with (PROCESSED_DIR / "data_report.json").open() as f:
    data_report = json.load(f)
print(json.dumps(data_report, indent=2)[:3000])


In [ ]:
# Shared - Multiclass-Safe Metrics
from sklearn.metrics import accuracy_score, average_precision_score, balanced_accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import label_binarize


def classification_metrics(y_true, y_proba, num_classes):
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=float)
    y_pred = y_proba.argmax(axis=1)
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }
    if num_classes == 2:
        metrics["roc_auc"] = float(roc_auc_score(y_true, y_proba[:, 1]))
        metrics["pr_auc"] = float(average_precision_score(y_true, y_proba[:, 1]))
    else:
        y_bin = label_binarize(y_true, classes=list(range(num_classes)))
        metrics["roc_auc_ovr"] = float(roc_auc_score(y_true, y_proba, multi_class="ovr", average="macro"))
        metrics["macro_pr_auc"] = float(average_precision_score(y_bin, y_proba, average="macro"))
    return metrics


In [ ]:
# Cell 4 - CoLES Runtime Knobs
COLES_ARTICLE_LIKE_PREP = True
COLES_PRETRAIN_ENTITY_SCOPE = "train_prefix_only"
COLES_DOWNSTREAM_MODES = ["classification_head", "full_finetune", "catboost"]
COLES_PRETRAIN_EPOCHS = 20
COLES_FINETUNE_EPOCHS = 25
COLES_BATCH_SIZE = 128
COLES_EMBED_DIM = 256
COLES_MIN_SUBSEQ_LEN = 8
COLES_MAX_SUBSEQ_LEN = 128
PREFIX_FRAC = 0.70
SMOKE_RUN = False

with (PROCESSED_DIR / "prepared_config.yaml").open() as f:
    cfg = yaml.safe_load(f)
canonical_df = pd.read_parquet(PROCESSED_DIR / "canonical_events.parquet")
with (PROCESSED_DIR / "splits.json").open() as f:
    splits = json.load(f)
entity_col = cfg["data"]["group_col"]
time_col = cfg["data"]["timestamp_col"]
target_col = cfg["data"]["target_col"]
event_col = cfg["data"]["event_type_col"]
num_cols = list(cfg["data"].get("numerical_cols") or [])
cat_cols = list(cfg["data"].get("categorical_cols") or [])
num_classes = int(cfg["training"].get("num_classes", 2))
print(canonical_df.shape, num_classes)


In [ ]:
# Cell 5 - Article-Like Prefix Slicing and Dataset

def sample_article_like_slice(group, rng, min_len=COLES_MIN_SUBSEQ_LEN, max_len=COLES_MAX_SUBSEQ_LEN):
    n = len(group)
    if n <= min_len:
        return group
    length = int(rng.integers(min_len, min(max_len, n) + 1))
    start = int(rng.integers(0, n - length + 1))
    return group.iloc[start:start + length]


def manual_article_like_multi_slice(group, rng, n_views=2):
    return [sample_article_like_slice(group, rng) for _ in range(n_views)]


def prefix_only(df):
    chunks = []
    meta = []
    for _, group in df.sort_values([entity_col, time_col], kind="stable").groupby(entity_col, sort=False):
        cutoff = max(1, min(len(group), int(np.ceil(len(group) * PREFIX_FRAC))))
        chunks.append(group.iloc[:cutoff].copy())
        meta.append({entity_col: group[entity_col].iloc[0], "prefix_events": cutoff, "full_events": len(group)})
    return pd.concat(chunks, ignore_index=True), pd.DataFrame(meta)


def assert_no_future_leakage(meta):
    assert (meta["prefix_events"] <= meta["full_events"]).all()
    assert (meta["prefix_events"] >= 1).all()

train_prefix, train_meta = prefix_only(canonical_df[canonical_df[entity_col].isin(splits["train"])])
assert_no_future_leakage(train_meta)


In [ ]:
# Cell 6 - Minimal CoLES Encoder and Loss
# This notebook keeps the article-like sampling protocol in-notebook.  If PTLS is
# installed, you can replace this compact encoder with ptls.nn.TrxEncoder + RnnSeqEncoder.
class PrefixSequenceDataset(Dataset):
    def __init__(self, df, ids, pretrain=False, seed=42):
        self.df = df[df[entity_col].isin(ids)].sort_values([entity_col, time_col], kind="stable")
        self.groups = [g for _, g in self.df.groupby(entity_col, sort=False)]
        self.pretrain = pretrain
        self.rng = np.random.default_rng(seed)
        events = canonical_df[event_col].astype(str).unique().tolist()
        self.event_vocab = {v: i + 1 for i, v in enumerate(sorted(events))}

    def __len__(self):
        return len(self.groups)

    def _encode_group(self, group):
        ids = [self.event_vocab.get(str(x), 0) for x in group[event_col].tolist()]
        return torch.tensor(ids, dtype=torch.long)

    def __getitem__(self, idx):
        group = self.groups[idx]
        label = int(group[target_col].iloc[0])
        if self.pretrain and COLES_ARTICLE_LIKE_PREP:
            views = manual_article_like_multi_slice(group, self.rng, n_views=2)
            return [self._encode_group(v) for v in views], label
        return self._encode_group(group), label


def pad_1d(seqs):
    max_len = max(len(s) for s in seqs)
    out = torch.zeros(len(seqs), max_len, dtype=torch.long)
    mask = torch.zeros(len(seqs), max_len, dtype=torch.bool)
    for i, s in enumerate(seqs):
        out[i, :len(s)] = s
        mask[i, :len(s)] = True
    return out, mask


def coles_pretrain_collate(batch):
    view1 = [item[0][0] for item in batch]
    view2 = [item[0][1] for item in batch]
    x1, m1 = pad_1d(view1)
    x2, m2 = pad_1d(view2)
    return x1, m1, x2, m2


def supervised_collate(batch):
    seqs = [item[0] for item in batch]
    labels = torch.tensor([item[1] for item in batch], dtype=torch.long)
    x, m = pad_1d(seqs)
    return x, m, labels

class CoLESClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        self.emb = nn.Embedding(vocab_size + 1, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, embed_dim // 2, batch_first=True, bidirectional=True)
        self.head = nn.Linear(embed_dim, num_classes)

    def encode(self, x, mask):
        emb = self.emb(x)
        out, _ = self.gru(emb)
        mask_f = mask.unsqueeze(-1).float()
        return (out * mask_f).sum(1) / mask_f.sum(1).clamp_min(1.0)

    def forward(self, x, mask):
        return self.head(self.encode(x, mask))


def contrastive_loss(z1, z2, temperature=0.07):
    z1 = torch.nn.functional.normalize(z1, dim=1)
    z2 = torch.nn.functional.normalize(z2, dim=1)
    logits = z1 @ z2.T / temperature
    labels = torch.arange(z1.size(0), device=z1.device)
    return (torch.nn.functional.cross_entropy(logits, labels) + torch.nn.functional.cross_entropy(logits.T, labels)) / 2


def make_class_weights(labels):
    values = np.asarray(labels, dtype=int)
    counts = np.bincount(values, minlength=num_classes).astype(float)
    weights = counts.sum() / np.maximum(counts, 1.0)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32)


In [ ]:
# Cell 7 - CoLES Pretrain, CoLES + Classification Head, CoLES Full Fine-Tune
import copy
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_ds_pre = PrefixSequenceDataset(train_prefix, splits["train"], pretrain=True)
vocab_size = len(train_ds_pre.event_vocab)
model = CoLESClassifier(vocab_size, COLES_EMBED_DIM, num_classes).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
pre_loader = DataLoader(train_ds_pre, batch_size=COLES_BATCH_SIZE, shuffle=True, collate_fn=coles_pretrain_collate)

pre_epochs = 1 if SMOKE_RUN else COLES_PRETRAIN_EPOCHS
for epoch in range(pre_epochs):
    model.train()
    losses = []
    for x1, m1, x2, m2 in pre_loader:
        x1, m1, x2, m2 = x1.to(device), m1.to(device), x2.to(device), m2.to(device)
        loss = contrastive_loss(model.encode(x1, m1), model.encode(x2, m2))
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(float(loss.detach().cpu()))
    print(epoch, np.mean(losses))

pretrained_state = copy.deepcopy(model.state_dict())

# Downstream train/eval slices are prefix-only to avoid future leakage.
train_downstream_prefix, train_downstream_meta = prefix_only(canonical_df[canonical_df[entity_col].isin(splits["train"])])
val_downstream_prefix, val_downstream_meta = prefix_only(canonical_df[canonical_df[entity_col].isin(splits["val"])])
test_downstream_prefix, test_downstream_meta = prefix_only(canonical_df[canonical_df[entity_col].isin(splits["test"])])
assert_no_future_leakage(train_downstream_meta)
assert_no_future_leakage(val_downstream_meta)
assert_no_future_leakage(test_downstream_meta)
train_ds = PrefixSequenceDataset(train_downstream_prefix, splits["train"], pretrain=False)
val_ds = PrefixSequenceDataset(val_downstream_prefix, splits["val"], pretrain=False)
test_ds = PrefixSequenceDataset(test_downstream_prefix, splits["test"], pretrain=False)
train_loader = DataLoader(train_ds, batch_size=COLES_BATCH_SIZE, shuffle=True, collate_fn=supervised_collate)
val_loader = DataLoader(val_ds, batch_size=COLES_BATCH_SIZE, shuffle=False, collate_fn=supervised_collate)
test_loader = DataLoader(test_ds, batch_size=COLES_BATCH_SIZE, shuffle=False, collate_fn=supervised_collate)
labels_for_weights = [int(g[target_col].iloc[0]) for g in train_ds.groups]
criterion = nn.CrossEntropyLoss(weight=make_class_weights(labels_for_weights).to(device))


def predict_loader(eval_model, loader):
    eval_model.eval(); probs = []; labels = []
    with torch.no_grad():
        for x, m, y in loader:
            p = torch.softmax(eval_model(x.to(device), m.to(device)), dim=-1).cpu().numpy()
            probs.append(p); labels.append(y.numpy())
    return np.concatenate(labels), np.concatenate(probs)


def train_supervised_downstream(freeze_encoder):
    downstream_model = CoLESClassifier(vocab_size, COLES_EMBED_DIM, num_classes).to(device)
    downstream_model.load_state_dict(pretrained_state)
    for module in [downstream_model.emb, downstream_model.gru]:
        for p in module.parameters():
            p.requires_grad = not freeze_encoder
    params = [p for p in downstream_model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=1e-4 if not freeze_encoder else 3e-4, weight_decay=1e-4)
    ft_epochs = 1 if SMOKE_RUN else COLES_FINETUNE_EPOCHS
    best_state, best_val = None, -1.0
    for epoch in range(ft_epochs):
        downstream_model.train()
        for x, m, y in train_loader:
            x, m, y = x.to(device), m.to(device), y.to(device)
            loss = criterion(downstream_model(x, m), y)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
        y_val, p_val = predict_loader(downstream_model, val_loader)
        val_score = classification_metrics(y_val, p_val, num_classes)["macro_f1"]
        if val_score > best_val:
            best_val = val_score
            best_state = copy.deepcopy(downstream_model.state_dict())
    downstream_model.load_state_dict(best_state)
    y_test, p_test = predict_loader(downstream_model, test_loader)
    return classification_metrics(y_test, p_test, num_classes), downstream_model

head_metrics, head_model = train_supervised_downstream(freeze_encoder=True)
full_metrics, full_model = train_supervised_downstream(freeze_encoder=False)
{"coles_classification_head": head_metrics, "coles_full_finetune": full_metrics}

In [ ]:
# Cell 8 - Extract CoLES Embeddings and Train CatBoost

def extract_coles_embeddings(dataset, encoder_model):
    loader = DataLoader(dataset, batch_size=COLES_BATCH_SIZE, shuffle=False, collate_fn=supervised_collate)
    encoder_model.eval(); embs = []; labels = []
    with torch.no_grad():
        for x, m, y in loader:
            z = encoder_model.encode(x.to(device), m.to(device)).cpu().numpy()
            embs.append(z); labels.append(y.numpy())
    return np.concatenate(embs), np.concatenate(labels), [g[entity_col].iloc[0] for g in dataset.groups]


def train_coles_catboost(encoder_model):
    X_train, y_train, _ = extract_coles_embeddings(train_ds, encoder_model)
    X_val, y_val, _ = extract_coles_embeddings(val_ds, encoder_model)
    X_test, y_test, _ = extract_coles_embeddings(test_ds, encoder_model)
    cb = CatBoostClassifier(
        iterations=100 if SMOKE_RUN else 2000,
        learning_rate=0.03,
        depth=5,
        loss_function="Logloss" if num_classes == 2 else "MultiClass",
        auto_class_weights="Balanced",
        random_seed=42,
        verbose=100,
    )
    cb.fit(Pool(X_train, y_train), eval_set=Pool(X_val, y_val), use_best_model=True)
    return cb, y_test, cb.predict_proba(X_test)

catboost_encoder = CoLESClassifier(vocab_size, COLES_EMBED_DIM, num_classes).to(device)
catboost_encoder.load_state_dict(pretrained_state)
cb_model, y_cb, p_cb = train_coles_catboost(catboost_encoder)
catboost_metrics = classification_metrics(y_cb, p_cb, num_classes)
catboost_metrics

In [ ]:
# Cell 9 - Save CoLES Outputs
out_dir = OUTPUT_ROOT / DATASET_NAME / "coles"
out_dir.mkdir(parents=True, exist_ok=True)
summary = {
    "coles_classification_head": head_metrics,
    "coles_full_finetune": full_metrics,
    "coles_catboost": catboost_metrics,
}
with (out_dir / "coles_metrics.json").open("w") as f:
    json.dump(summary, f, indent=2)
fi = pd.DataFrame({"feature": [f"emb_{i}" for i in range(COLES_EMBED_DIM)], "importance": cb_model.get_feature_importance()})
fi.to_csv(out_dir / "coles_catboost_feature_importance.csv", index=False)
print(out_dir / "coles_metrics.json")
summary